# Boundless Prover Stats

> Source: https://github.com/zerokrab/boundless-jupyter

Analyze your prover's order history and mining performance on the [Boundless Network](https://explorer.boundless.network).

This notebook combines two views:
1. **Order History** — per-order breakdown and per-epoch summaries of fulfilled vs expired orders and cycle counts.
2. **Mining vs Market Comparison** — how much of your mining capacity is attributed to market orders vs proof-of-valid-work (PoVW) mining rewards.

## Where to run

| Environment | Works? | Notes |
|-------------|--------|-------|
| [jupyter.zerokrab.com](https://jupyter.zerokrab.com) | ✅ | Hosted JupyterLite — CORS proxy handles API requests |
| Standard Jupyter (local) | ✅ | `jupyter lab` / `jupyter notebook` — goes direct, no CORS |
| JupyterLite locally | ❌ | `localhost` not in proxy’s allowed origins — use standard Jupyter instead |

> **Running locally?** Use standard Jupyter (`pip install jupyterlab && jupyter lab`), not JupyterLite. See the [README](https://github.com/zerokrab/boundless-jupyter#readme) for setup instructions.

## How to run

1. Edit the **Configuration** cell below — set your `PROVER_ADDRESS` and optionally `MINER_LOG_ID`.
2. In the top menu select **Run → Run All Cells**.
3. The mining comparison section will be skipped automatically if `MINER_LOG_ID` is left blank.

In [ ]:
# ── Configuration ──────────────────────────────────────────────
PROVER_ADDRESS = "0xYourProverAddressHere"   # Required
MINER_LOG_ID   = ""                           # Optional: fill in for mining comparison
EPOCH_START    = None  # e.g. 1  — set to an int to filter epochs (inclusive)
EPOCH_END      = None  # e.g. 10 — set to an int to filter epochs (inclusive)
# ───────────────────────────────────────────────────────────────

In [ ]:
import json
import sys

_IN_PYODIDE = "pyodide" in sys.modules

# When running in JupyterLite (hosted), use the CORS proxy.
# When running locally in standard Jupyter, go direct (no browser CORS restrictions).
BASE_URL = "https://proxy.jupyter.zerokrab.com/api" if _IN_PYODIDE else "https://explorer.boundless.network/api"

async def fetch_json(url):
    """Fetch a URL and return parsed JSON.
    Uses pyodide.http.pyfetch (async Fetch API) when running in JupyterLite/Pyodide,
    or urllib.request when running in standard Jupyter.
    """
    if _IN_PYODIDE:
        from pyodide.http import pyfetch
        resp = await pyfetch(url)
        return await resp.json()
    else:
        import urllib.request
        with urllib.request.urlopen(url) as r:
            return json.loads(r.read().decode())

async def fetch_all_orders(address, page_size=200):
    """Paginate through /api/provers/{address}/orders until all records fetched."""
    all_orders = []
    skip = 0
    while True:
        url = f"{BASE_URL}/provers/{address}/orders?limit={page_size}&skip={skip}"
        data = await fetch_json(url)
        # API may return a list directly or wrap in {orders: [...]} / {data: [...]}
        if isinstance(data, list):
            batch = data
        elif isinstance(data, dict):
            batch = data.get("orders") or data.get("data") or data.get("results") or []
        else:
            batch = []
        if not batch:
            break
        all_orders.extend(batch)
        if len(batch) < page_size:
            break
        skip += page_size
    return all_orders

async def fetch_epoch_aggregates(address):
    """Fetch per-epoch aggregate summary for the prover."""
    url = f"{BASE_URL}/provers/{address}/aggregates/epoch"
    data = await fetch_json(url)
    if isinstance(data, list):
        return data
    return data.get("epochs") or data.get("data") or data.get("results") or []

async def fetch_miner_data(log_id):
    """Fetch mining data for a given miner log ID."""
    url = f"{BASE_URL}/miners/{log_id}"
    data = await fetch_json(url)
    if isinstance(data, list):
        return data
    return data.get("epochs") or data.get("data") or data.get("results") or [data]

print("✅ Fetch helpers ready.")

In [ ]:
import pandas as pd

print(f"Fetching orders for prover: {PROVER_ADDRESS} …")
raw_orders = await fetch_all_orders(PROVER_ADDRESS)
print(f"  → {len(raw_orders)} orders fetched.")

print("Fetching epoch aggregates …")
raw_epochs = await fetch_epoch_aggregates(PROVER_ADDRESS)
print(f"  → {len(raw_epochs)} epoch records fetched.")

# ── Build orders DataFrame ────────────────────────────────────
df_orders = pd.DataFrame(raw_orders)
if df_orders.empty:
    print("⚠️  No order data returned. Check your PROVER_ADDRESS.")
else:
    # Normalise column names to lowercase
    df_orders.columns = [c.lower() for c in df_orders.columns]
    # Identify epoch, status, cycles columns (handle API variations)
    epoch_col  = next((c for c in df_orders.columns if 'epoch' in c), None)
    status_col = next((c for c in df_orders.columns if 'status' in c), None)
    cycles_col = next((c for c in df_orders.columns if 'cycle' in c), None)
    # Apply epoch filter
    if epoch_col and EPOCH_START is not None:
        df_orders = df_orders[df_orders[epoch_col] >= EPOCH_START]
    if epoch_col and EPOCH_END is not None:
        df_orders = df_orders[df_orders[epoch_col] <= EPOCH_END]
    print(f"\nOrders after epoch filter: {len(df_orders)}")

# ── Build epoch aggregates DataFrame ─────────────────────────
df_epochs = pd.DataFrame(raw_epochs)
if not df_epochs.empty:
    df_epochs.columns = [c.lower() for c in df_epochs.columns]
    ep_epoch_col = next((c for c in df_epochs.columns if 'epoch' in c), None)
    if ep_epoch_col and EPOCH_START is not None:
        df_epochs = df_epochs[df_epochs[ep_epoch_col] >= EPOCH_START]
    if ep_epoch_col and EPOCH_END is not None:
        df_epochs = df_epochs[df_epochs[ep_epoch_col] <= EPOCH_END]

print("✅ Data loaded.")

## Order History

In [ ]:
from IPython.display import display

if df_orders.empty:
    print("No orders to display.")
else:
    # Surface the most useful columns first
    priority_cols = [epoch_col, status_col, cycles_col]
    priority_cols = [c for c in priority_cols if c is not None]
    other_cols    = [c for c in df_orders.columns if c not in priority_cols]
    display_cols  = priority_cols + other_cols
    display(df_orders[display_cols].reset_index(drop=True))

## Epoch Summary

In [ ]:
# Build epoch summary from orders if the API aggregate endpoint didn't return useful data
if not df_orders.empty and epoch_col and status_col:
    fulfilled_mask = df_orders[status_col].astype(str).str.lower().str.contains('fulfill|complete|success')
    expired_mask   = df_orders[status_col].astype(str).str.lower().str.contains('expire|fail|cancel')

    grp = df_orders.groupby(epoch_col)
    epoch_summary = pd.DataFrame({
        'total_orders':    grp.size(),
        'fulfilled_orders': grp.apply(lambda x: fulfilled_mask.loc[x.index].sum()),
        'expired_orders':   grp.apply(lambda x: expired_mask.loc[x.index].sum()),
    })
    if cycles_col:
        epoch_summary['total_cycles'] = grp[cycles_col].sum()
    epoch_summary = epoch_summary.reset_index().rename(columns={epoch_col: 'epoch'})
    print("Epoch summary (derived from orders):")
    display(epoch_summary)
elif not df_epochs.empty:
    print("Epoch summary (from API aggregates):")
    display(df_epochs.reset_index(drop=True))
else:
    print("No epoch data available.")

## Visualizations — Orders

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

%matplotlib inline

if df_orders.empty:
    print("No order data to visualize.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f"Order History — {PROVER_ADDRESS[:10]}…", fontsize=13, fontweight='bold')

    # ── Chart 1: Cycles per epoch (bar) ──────────────────────────
    ax1 = axes[0]
    if epoch_col and cycles_col:
        cycles_by_epoch = df_orders.groupby(epoch_col)[cycles_col].sum().sort_index()
        ax1.bar(cycles_by_epoch.index.astype(str), cycles_by_epoch.values, color='steelblue', edgecolor='white')
        ax1.set_title('Total Cycles per Epoch')
        ax1.set_xlabel('Epoch')
        ax1.set_ylabel('Cycles')
        ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M' if x >= 1e6 else f'{x:.0f}'))
        ax1.tick_params(axis='x', rotation=45)
    else:
        ax1.text(0.5, 0.5, 'epoch or cycles column not found', ha='center', va='center', transform=ax1.transAxes)
        ax1.set_title('Cycles per Epoch')

    # ── Chart 2: Fulfilled vs Expired (pie) ──────────────────────
    ax2 = axes[1]
    if status_col:
        status_counts = df_orders[status_col].astype(str).str.lower().value_counts()
        # Group into fulfilled / expired / other
        def classify(s):
            if any(k in s for k in ('fulfill', 'complete', 'success')):
                return 'Fulfilled'
            elif any(k in s for k in ('expire', 'fail', 'cancel')):
                return 'Expired / Failed'
            return 'Other'
        grouped = df_orders[status_col].astype(str).str.lower().map(classify).value_counts()
        colors  = {'Fulfilled': '#2ecc71', 'Expired / Failed': '#e74c3c', 'Other': '#95a5a6'}
        wedge_colors = [colors.get(k, '#bdc3c7') for k in grouped.index]
        ax2.pie(
            grouped.values,
            labels=grouped.index,
            colors=wedge_colors,
            autopct='%1.1f%%',
            startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 1.5}
        )
        ax2.set_title('Order Outcomes')
    else:
        ax2.text(0.5, 0.5, 'status column not found', ha='center', va='center', transform=ax2.transAxes)
        ax2.set_title('Order Outcomes')

    plt.tight_layout()
    plt.show()

## Mining vs Market Comparison

> This section only runs when `MINER_LOG_ID` is set in the Configuration cell.

In [ ]:
if not MINER_LOG_ID:
    print("ℹ️  MINER_LOG_ID is not set — skipping mining comparison.")
    _skip_mining = True
else:
    _skip_mining = False
    print(f"Fetching mining data for log ID: {MINER_LOG_ID} …")
    raw_mining = await fetch_miner_data(MINER_LOG_ID)
    print(f"  → {len(raw_mining)} mining epoch records fetched.")
    df_mining = pd.DataFrame(raw_mining)
    if df_mining.empty:
        print("⚠️  No mining data returned. Check your MINER_LOG_ID.")
        _skip_mining = True
    else:
        df_mining.columns = [c.lower() for c in df_mining.columns]
        m_epoch_col  = next((c for c in df_mining.columns if 'epoch' in c), None)
        m_cycles_col = next((c for c in df_mining.columns if 'cycle' in c), None)
        m_reward_col = next((c for c in df_mining.columns if 'reward' in c), None)
        if m_epoch_col and EPOCH_START is not None:
            df_mining = df_mining[df_mining[m_epoch_col] >= EPOCH_START]
        if m_epoch_col and EPOCH_END is not None:
            df_mining = df_mining[df_mining[m_epoch_col] <= EPOCH_END]
        print(f"  → Mining data after epoch filter: {len(df_mining)} rows.")
        print("✅ Mining data loaded.")

In [ ]:
if _skip_mining:
    print("Mining comparison skipped.")
else:
    # Derive market cycles per epoch from orders
    if not df_orders.empty and epoch_col and cycles_col:
        market_cycles_by_epoch = df_orders.groupby(epoch_col)[cycles_col].sum().rename('market_cycles')
    else:
        market_cycles_by_epoch = pd.Series(dtype=float, name='market_cycles')

    # Merge mining + market
    if m_epoch_col and m_cycles_col:
        df_compare = df_mining[[m_epoch_col, m_cycles_col]].copy()
        df_compare = df_compare.rename(columns={m_epoch_col: 'epoch', m_cycles_col: 'mining_cycles'})
        if m_reward_col:
            df_compare['rewards'] = df_mining[m_reward_col].values
        df_compare = df_compare.set_index('epoch').join(market_cycles_by_epoch, how='left')
        df_compare['market_cycles'] = df_compare['market_cycles'].fillna(0)
        df_compare['market_pct'] = (
            df_compare['market_cycles'] / df_compare['mining_cycles'].replace(0, float('nan')) * 100
        ).round(2)
        df_compare = df_compare.reset_index()
        print("Market vs Mining comparison per epoch:")
        display(df_compare)
    else:
        print("⚠️  Could not identify epoch or cycles columns in mining data.")
        display(df_mining)

## Visualizations — Mining

In [ ]:
if _skip_mining:
    print("Mining visualizations skipped (MINER_LOG_ID not set).")
elif 'df_compare' not in dir() or df_compare.empty:
    print("No comparison data to visualize.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f"Mining vs Market — {PROVER_ADDRESS[:10]}…", fontsize=13, fontweight='bold')

    epochs = df_compare['epoch'].astype(str)
    x      = np.arange(len(epochs))

    # ── Chart 1: Stacked bar — market vs non-market cycles ───────
    ax1 = axes[0]
    if 'mining_cycles' in df_compare.columns and 'market_cycles' in df_compare.columns:
        non_market = (df_compare['mining_cycles'] - df_compare['market_cycles']).clip(lower=0)
        market     = df_compare['market_cycles']
        bar_w = 0.6
        ax1.bar(x, non_market, bar_w, label='PoVW / Non-market', color='#3498db', edgecolor='white')
        ax1.bar(x, market,     bar_w, bottom=non_market, label='Market orders', color='#e67e22', edgecolor='white')
        ax1.set_xticks(x)
        ax1.set_xticklabels(epochs, rotation=45, ha='right')
        ax1.set_title('Cycles per Epoch (Stacked)')
        ax1.set_xlabel('Epoch')
        ax1.set_ylabel('Cycles')
        ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v/1e6:.1f}M' if v >= 1e6 else f'{v:.0f}'))
        ax1.legend(loc='upper left', fontsize=9)
    else:
        ax1.text(0.5, 0.5, 'cycle columns not available', ha='center', va='center', transform=ax1.transAxes)

    # ── Chart 2: Line — market % over epochs ────────────────────
    ax2 = axes[1]
    if 'market_pct' in df_compare.columns:
        ax2.plot(x, df_compare['market_pct'], marker='o', color='#e67e22', linewidth=2, markersize=6)
        ax2.fill_between(x, df_compare['market_pct'], alpha=0.15, color='#e67e22')
        ax2.axhline(df_compare['market_pct'].mean(), linestyle='--', color='gray', linewidth=1, label=f"Mean: {df_compare['market_pct'].mean():.1f}%")
        ax2.set_xticks(x)
        ax2.set_xticklabels(epochs, rotation=45, ha='right')
        ax2.set_title('Market Order Share (%) over Epochs')
        ax2.set_xlabel('Epoch')
        ax2.set_ylabel('Market %')
        ax2.set_ylim(0, max(df_compare['market_pct'].max() * 1.2, 10))
        ax2.legend(fontsize=9)
    else:
        ax2.text(0.5, 0.5, 'market_pct not available', ha='center', va='center', transform=ax2.transAxes)

    plt.tight_layout()
    plt.show()